In [0]:
%run /Workspace/Shared/ADC-DB/Prod/Pipelines/Blob/blob_shared_lib

In [0]:
import json
import math
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark import StorageLevel
from pyspark.sql import Row, functions as F, types as T
from pyspark.sql.window import Window


dbutils.widgets.text("RUN_ID", "")
dbutils.widgets.text("SHARDS", "8")
dbutils.widgets.text("SHARD_ID", "")
dbutils.widgets.text("MAX_ROWS_PER_SHARD", "10000")
dbutils.widgets.text("MAX_CONCURRENT_SHARDS", "4")
dbutils.widgets.text("SHARD_TIMEOUT_SEC", "21600")
dbutils.widgets.text("METADATA_TABLE", "4_prod.tmp.blob_pipeline_metadata_v3")
dbutils.widgets.text("HISTORY_TABLE", "4_prod.logs.mill_blob_extractor_history")
dbutils.widgets.text("STAGING_DB", "4_prod.tmp")
dbutils.widgets.text("PROCESSING_CONFIG_TABLE", "4_prod.tmp.blob_processing_config_v3")

In [0]:


RUN_ID = dbutils.widgets.get("RUN_ID").strip()
SHARDS = int(dbutils.widgets.get("SHARDS"))
SHARD_ID_RAW = dbutils.widgets.get("SHARD_ID").strip()
MAX_ROWS_PER_SHARD = int(dbutils.widgets.get("MAX_ROWS_PER_SHARD"))
MAX_CONCURRENT_SHARDS = int(dbutils.widgets.get("MAX_CONCURRENT_SHARDS"))
SHARD_TIMEOUT_SEC = int(dbutils.widgets.get("SHARD_TIMEOUT_SEC"))
METADATA_TABLE = dbutils.widgets.get("METADATA_TABLE").strip()
HISTORY_TABLE = dbutils.widgets.get("HISTORY_TABLE").strip()
STAGING_DB = dbutils.widgets.get("STAGING_DB").strip()
PROCESSING_CONFIG_TABLE = dbutils.widgets.get("PROCESSING_CONFIG_TABLE").strip()

IS_ORCHESTRATOR = SHARD_ID_RAW == ""
SHARD_ID = None if IS_ORCHESTRATOR else int(SHARD_ID_RAW)

if not RUN_ID:
    raise ValueError("RUN_ID is required; implicit latest-run selection is intentionally disabled")
if not re.fullmatch(r"[A-Za-z0-9_.:-]+", RUN_ID):
    raise ValueError("Unsafe RUN_ID")
if not 1 <= SHARDS <= 128:
    raise ValueError("SHARDS must be between 1 and 128")
if SHARD_ID is not None and not 0 <= SHARD_ID < SHARDS:
    raise ValueError("SHARD_ID is outside the configured shard range")
if MAX_ROWS_PER_SHARD < 1:
    raise ValueError("MAX_ROWS_PER_SHARD must be positive")
if not 1 <= MAX_CONCURRENT_SHARDS <= 32:
    raise ValueError("MAX_CONCURRENT_SHARDS must be between 1 and 32")
if SHARD_TIMEOUT_SEC < 60:
    raise ValueError("SHARD_TIMEOUT_SEC must be at least 60")

safe_run_id = re.sub(r"[^A-Za-z0-9_]", "_", RUN_ID)


def validate_table_name(value: str) -> str:
    if not re.fullmatch(r"[A-Za-z0-9_]+(?:\.[A-Za-z0-9_]+){1,2}", value):
        raise ValueError(f"Unsafe table identifier: {value!r}")
    return value


def quote_table(value: str) -> str:
    validate_table_name(value)
    tick = chr(96)
    return ".".join(f"{tick}{part}{tick}" for part in value.split("."))


for table_name in (METADATA_TABLE, HISTORY_TABLE, STAGING_DB, PROCESSING_CONFIG_TABLE):
    validate_table_name(table_name)

try:
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
    spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
except Exception as exc:
    print(f"Non-fatal Spark configuration warning: {exc}")


def metadata_row():
    rows = spark.table(METADATA_TABLE).filter(F.col("run_id") == RUN_ID).limit(2).collect()
    if len(rows) != 1:
        raise RuntimeError(f"Expected exactly one metadata row for {RUN_ID}; found {len(rows)}")
    return rows[0]


def sql_string(value: str) -> str:
    return "'" + (value or "").replace("'", "''") + "'"


def ensure_run_config():
    spark.sql(f"""
      CREATE TABLE IF NOT EXISTS {quote_table(PROCESSING_CONFIG_TABLE)} (
        run_id STRING NOT NULL,
        shards INT NOT NULL,
        max_rows_per_shard INT NOT NULL,
        created_ts TIMESTAMP,
        updated_ts TIMESTAMP
      ) USING DELTA
    """)
    rows = (
        spark.table(PROCESSING_CONFIG_TABLE)
        .filter(F.col("run_id") == RUN_ID)
        .limit(2)
        .collect()
    )
    if len(rows) > 1:
        raise RuntimeError(f"Duplicate processing config rows for {RUN_ID}")
    if rows:
        configured_shards = int(rows[0]["shards"])
        if configured_shards != SHARDS:
            raise RuntimeError(
                f"RUN_ID {RUN_ID} was initialized with SHARDS={configured_shards}; "
                f"cannot resume it with SHARDS={SHARDS}"
            )
        spark.sql(f"""
          UPDATE {quote_table(PROCESSING_CONFIG_TABLE)}
          SET max_rows_per_shard = {MAX_ROWS_PER_SHARD},
              updated_ts = current_timestamp()
          WHERE run_id = {sql_string(RUN_ID)}
        """)
    else:
        spark.sql(f"""
          INSERT INTO {quote_table(PROCESSING_CONFIG_TABLE)}
            (run_id, shards, max_rows_per_shard, created_ts, updated_ts)
          VALUES
            ({sql_string(RUN_ID)}, {SHARDS}, {MAX_ROWS_PER_SHARD},
             current_timestamp(), current_timestamp())
        """)


In [0]:
if IS_ORCHESTRATOR:
    meta = metadata_row()
    if meta["status"] not in {
        "worklist_created", "processing_failed", "processing",
        "processing_partial",
    }:
        raise RuntimeError(f"Run {RUN_ID} is in non-processable status {meta['status']!r}")

    WORKLIST_TABLE = meta["worklist_table"]
    expected_events = int(meta["total_events"] or 0)
    if not WORKLIST_TABLE or not spark.catalog.tableExists(WORKLIST_TABLE):
        raise RuntimeError(f"Worklist does not exist for run {RUN_ID}: {WORKLIST_TABLE}")

    ensure_run_config()

    spark.sql(f"""
      UPDATE {quote_table(METADATA_TABLE)}
      SET status = 'processing', error_message = NULL
      WHERE run_id = {sql_string(RUN_ID)}
    """)

    notebook_path = (
        dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    )

    def run_shard(shard_id: int):
        try:
            raw_result = dbutils.notebook.run(
                notebook_path,
                SHARD_TIMEOUT_SEC,
                {
                    "RUN_ID": RUN_ID,
                    "SHARDS": str(SHARDS),
                    "SHARD_ID": str(shard_id),
                    "MAX_ROWS_PER_SHARD": str(MAX_ROWS_PER_SHARD),
                    "MAX_CONCURRENT_SHARDS": str(MAX_CONCURRENT_SHARDS),
                    "SHARD_TIMEOUT_SEC": str(SHARD_TIMEOUT_SEC),
                    "METADATA_TABLE": METADATA_TABLE,
                    "HISTORY_TABLE": HISTORY_TABLE,
                    "STAGING_DB": STAGING_DB,
                    "PROCESSING_CONFIG_TABLE": PROCESSING_CONFIG_TABLE,
                },
            )
            payload = json.loads(raw_result)
            if payload.get("status") != "success":
                raise RuntimeError(raw_result)
            return payload
        except Exception as exc:
            return {
                "status": "failed",
                "shard_id": shard_id,
                "error": f"{type(exc).__name__}: {str(exc)[:1000]}",
            }

    started = time.time()
    results = []
    worker_count = min(SHARDS, MAX_CONCURRENT_SHARDS)
    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        futures = {executor.submit(run_shard, shard): shard for shard in range(SHARDS)}
        for future in as_completed(futures):
            result = future.result()
            results.append(result)
            print(
                f"shard {result['shard_id']}: {result['status']} "
                f"selected={result.get('selected_events', 'n/a')}"
            )

    failures = [result for result in results if result["status"] != "success"]
    if failures:
        error_summary = json.dumps(failures, sort_keys=True)[:8000]
        spark.sql(f"""
          UPDATE {quote_table(METADATA_TABLE)}
          SET status = 'processing_failed',
              error_message = {sql_string(error_summary)}
          WHERE run_id = {sql_string(RUN_ID)}
        """)
        raise RuntimeError(f"{len(failures)} shard(s) failed: {error_summary}")

    results.sort(key=lambda item: item["shard_id"])
    batch_tables = [item["batch_table"] for item in results]
    history_tables = [item["history_table"] for item in results]
    missing_tables = [
        table for table in batch_tables + history_tables
        if not spark.catalog.tableExists(table)
    ]
    if missing_tables:
        raise RuntimeError(f"Successful shards did not leave expected tables: {missing_tables}")

    batch_count = sum(int(spark.table(table).count()) for table in batch_tables)
    history_count = sum(int(spark.table(table).count()) for table in history_tables)
    remaining = int(
        spark.table(WORKLIST_TABLE)
        .filter(F.col("status") != "completed")
        .count()
    )
    processed_this_invocation = sum(int(item["selected_events"]) for item in results)

    common_update = f"""
        batch_tables = {sql_string(','.join(batch_tables))},
        history_tables = {sql_string(','.join(history_tables))},
        processed_events = {batch_count},
        error_message = NULL
    """

    if remaining:
        spark.sql(f"""
          UPDATE {quote_table(METADATA_TABLE)}
          SET {common_update},
              status = 'processing_partial',
              completed_ts = NULL
          WHERE run_id = {sql_string(RUN_ID)}
        """)
        result = {
            "status": "partial",
            "run_id": RUN_ID,
            "processed_this_invocation": processed_this_invocation,
            "processed_total": batch_count,
            "remaining": remaining,
            "expected": expected_events,
            "has_more": True,
            "elapsed_seconds": round(time.time() - started, 1),
        }
        dbutils.jobs.taskValues.set(key="RUN_ID", value=RUN_ID)
        dbutils.jobs.taskValues.set(key="HAS_MORE_PROCESSING", value="true")
        dbutils.notebook.exit(json.dumps(result, sort_keys=True))

    if batch_count != expected_events or history_count != expected_events:
        details = {
            "expected": expected_events,
            "batch_count": batch_count,
            "history_count": history_count,
            "remaining_worklist": remaining,
        }
        message = json.dumps(details, sort_keys=True)
        spark.sql(f"""
          UPDATE {quote_table(METADATA_TABLE)}
          SET status = 'processing_failed',
              error_message = {sql_string(message)}
          WHERE run_id = {sql_string(RUN_ID)}
        """)
        raise RuntimeError(f"Final processing validation failed: {message}")

    spark.sql(f"""
      UPDATE {quote_table(METADATA_TABLE)}
      SET {common_update},
          status = 'processing_complete',
          completed_ts = current_timestamp()
      WHERE run_id = {sql_string(RUN_ID)}
    """)
    result = {
        "status": "success",
        "run_id": RUN_ID,
        "processed_this_invocation": processed_this_invocation,
        "processed_total": batch_count,
        "remaining": 0,
        "expected": expected_events,
        "has_more": False,
        "elapsed_seconds": round(time.time() - started, 1),
    }
    dbutils.jobs.taskValues.set(key="RUN_ID", value=RUN_ID)
    dbutils.jobs.taskValues.set(key="HAS_MORE_PROCESSING", value="false")
    dbutils.notebook.exit(json.dumps(result, sort_keys=True))


In [0]:
meta = metadata_row()
WORKLIST_TABLE = meta["worklist_table"]
TARGET_TABLE = meta["target_table"]
if not WORKLIST_TABLE or not spark.catalog.tableExists(WORKLIST_TABLE):
    raise RuntimeError(f"Missing worklist table: {WORKLIST_TABLE}")

config_rows = (
    spark.table(PROCESSING_CONFIG_TABLE)
    .filter(F.col("run_id") == RUN_ID)
    .limit(2)
    .collect()
)
if len(config_rows) != 1 or int(config_rows[0]["shards"]) != SHARDS:
    raise RuntimeError(f"Missing or incompatible processing configuration for {RUN_ID}")

batch_table = f"{STAGING_DB}.batch_v3_{safe_run_id}_shard_{SHARD_ID:04d}"
history_table = f"{STAGING_DB}.history_v3_{safe_run_id}_shard_{SHARD_ID:04d}"


_udf_schema = T.StructType([
    T.StructField("EVENT_ID", T.LongType()),
    T.StructField("VALID_UNTIL_DT_TM", T.TimestampType()),
    T.StructField("VALID_FROM_DT_TM", T.TimestampType()),
    T.StructField("UPDT_DT_TM", T.TimestampType()),
    T.StructField("UPDT_ID", T.LongType()),
    T.StructField("UPDT_TASK", T.LongType()),
    T.StructField("UPDT_CNT", T.LongType()),
    T.StructField("UPDT_APPLCTX", T.LongType()),
    T.StructField("LAST_UTC_TS", T.TimestampType()),
    T.StructField("ADC_UPDT", T.TimestampType()),
    T.StructField("BLOB_BINARY", T.BinaryType()),
    T.StructField("CONTENT_TYPE", T.StringType()),
    T.StructField("ENCODING", T.StringType()),
    T.StructField("BLOB_TEXT", T.StringType()),
    T.StructField("BINARY_SIZE", T.LongType()),
    T.StructField("TEXT_LENGTH", T.LongType()),
    T.StructField("STATUS", T.StringType()),
    T.StructField("anon_text", T.StringType()),
    T.StructField("ENCNTR_ID", T.LongType()),
    T.StructField("Trust", T.StringType()),
    T.StructField("raw_sha256", T.StringType()),
    T.StructField("decompressor_version", T.IntegerType()),
    T.StructField("parser_version", T.IntegerType()),
    T.StructField("post_processor_version", T.IntegerType()),
    T.StructField("history_status", T.StringType()),
    T.StructField("truncation_flag", T.BooleanType()),
    T.StructField("truncation_reason", T.StringType()),
    T.StructField("ftfy_explain", T.StringType()),
    T.StructField("decompression_strategy", T.StringType()),
    T.StructField("parse_status", T.StringType()),
    T.StructField("metrics", T.StringType()),
])


def _safe_int(value):
    if value is None or value == "":
        return None
    try:
        return int(value)
    except Exception:
        try:
            return int(float(str(value)))
        except Exception:
            return None


_POOL_STATE = {"pool": None}


def _worker_pool():
    if _POOL_STATE["pool"] is None:
        _POOL_STATE["pool"] = BlobProcessPool(timeout=PROCESS_TIMEOUT_SEC)
    return _POOL_STATE["pool"]


def _chunk_value(chunk, name):
    try:
        return chunk[name]
    except Exception:
        return getattr(chunk, name, None)


@F.udf(returnType=_udf_schema)
def process_blob_udf(event_id, adc_updt, chunks_data):
    raw_sha = None
    try:
        sorted_chunks = sorted(
            chunks_data or [],
            key=lambda chunk: (
                _chunk_value(chunk, "BLOB_SEQ_NUM") is None,
                _chunk_value(chunk, "BLOB_SEQ_NUM") or 0,
            ),
        )
        first = sorted_chunks[0] if sorted_chunks else None
        original_chunks = [
            {
                "BLOB_SEQ_NUM": _chunk_value(chunk, "BLOB_SEQ_NUM"),
                "BLOB_CONTENTS": _chunk_value(chunk, "BLOB_CONTENTS"),
            }
            for chunk in sorted_chunks
        ]
        raw_sha = raw_content_sha256(original_chunks)
        combined = combine_blob_chunks(original_chunks)
        compression_cd = _chunk_value(first, "COMPRESSION_CD") if first else None
        blob_length = _chunk_value(first, "BLOB_LENGTH") if first else None

        if len(combined) > MAX_COMPRESSED_BYTES:
            result = {
                "text": None, "content_type": None, "encoding": None,
                "binary_size": len(combined), "text_length": None,
                "status": f"Compressed too large: {len(combined)} bytes",
                "parse_status": "compressed_too_large",
                "truncation_flag": False, "truncation_reason": None,
                "ftfy_explain": None, "decompression_strategy": None,
                "metrics": {
                    "compressed_bytes": len(combined),
                    "chunk_count": len(sorted_chunks),
                },
            }
        else:
            result = _worker_pool().process(
                combined, compression_cd, len(sorted_chunks), blob_length
            )

        status = result.get("status") or "Processing worker error"
        return Row(
            EVENT_ID=_safe_int(event_id),
            VALID_UNTIL_DT_TM=_chunk_value(first, "VALID_UNTIL_DT_TM") if first else None,
            VALID_FROM_DT_TM=_chunk_value(first, "VALID_FROM_DT_TM") if first else None,
            UPDT_DT_TM=_chunk_value(first, "UPDT_DT_TM") if first else None,
            UPDT_ID=_safe_int(_chunk_value(first, "UPDT_ID")) if first else None,
            UPDT_TASK=_safe_int(_chunk_value(first, "UPDT_TASK")) if first else None,
            UPDT_CNT=_safe_int(_chunk_value(first, "UPDT_CNT")) if first else None,
            UPDT_APPLCTX=_safe_int(_chunk_value(first, "UPDT_APPLCTX")) if first else None,
            LAST_UTC_TS=_chunk_value(first, "LAST_UTC_TS") if first else None,
            ADC_UPDT=adc_updt,
            BLOB_BINARY=None,
            CONTENT_TYPE=result.get("content_type"),
            ENCODING=result.get("encoding"),
            BLOB_TEXT=result.get("text"),
            BINARY_SIZE=_safe_int(result.get("binary_size")),
            TEXT_LENGTH=_safe_int(result.get("text_length")),
            STATUS=status,
            anon_text=None,
            ENCNTR_ID=_safe_int(_chunk_value(first, "ENCNTR_ID")) if first else None,
            Trust=_chunk_value(first, "Trust") if first else None,
            raw_sha256=raw_sha,
            decompressor_version=DECOMPRESSOR_VERSION,
            parser_version=PARSER_VERSION,
            post_processor_version=POST_PROCESSOR_VERSION,
            history_status=status,
            truncation_flag=bool(result.get("truncation_flag", False)),
            truncation_reason=result.get("truncation_reason"),
            ftfy_explain=result.get("ftfy_explain"),
            decompression_strategy=result.get("decompression_strategy"),
            parse_status=result.get("parse_status"),
            metrics=json.dumps(result.get("metrics") or {}, default=str, sort_keys=True),
        )
    except Exception as exc:
        status = f"Processing error: {type(exc).__name__}"
        return Row(
            EVENT_ID=_safe_int(event_id), VALID_UNTIL_DT_TM=None,
            VALID_FROM_DT_TM=None, UPDT_DT_TM=None, UPDT_ID=None, UPDT_TASK=None,
            UPDT_CNT=None, UPDT_APPLCTX=None, LAST_UTC_TS=None, ADC_UPDT=adc_updt,
            BLOB_BINARY=None, CONTENT_TYPE=None, ENCODING=None, BLOB_TEXT=None,
            BINARY_SIZE=None, TEXT_LENGTH=None, STATUS=status, anon_text=None,
            ENCNTR_ID=None, Trust=None, raw_sha256=raw_sha,
            decompressor_version=DECOMPRESSOR_VERSION,
            parser_version=PARSER_VERSION,
            post_processor_version=POST_PROCESSOR_VERSION,
            history_status=status, truncation_flag=False, truncation_reason=None,
            ftfy_explain=None, decompression_strategy=None, parse_status="udf_error",
            metrics=json.dumps({
                "error": f"{type(exc).__name__}: {str(exc)[:500]}"
            }),
        )


def merge_output(source, target_table, condition):
    if spark.catalog.tableExists(target_table):
        (
            DeltaTable.forName(spark, target_table)
            .alias("t")
            .merge(source.alias("s"), condition)
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    else:
        source.write.mode("overwrite").saveAsTable(target_table)


In [0]:
shard_predicate = (
    F.pmod(F.xxhash64("EVENT_ID", "ADC_UPDT"), F.lit(SHARDS)) == F.lit(SHARD_ID)
)
pending = (
    spark.table(WORKLIST_TABLE)
    .filter(F.col("status").isin("pending", "compressed_oversized"))
    .filter(shard_predicate)
)

selection_window = Window.orderBy(
    F.col("source_commit_version"),
    F.col("EVENT_ID"),
    F.col("ADC_UPDT").asc_nulls_first(),
)
events = (
    pending
    .withColumn("_batch_rank", F.row_number().over(selection_window))
    .filter(F.col("_batch_rank") <= F.lit(MAX_ROWS_PER_SHARD))
    .select("EVENT_ID", "ADC_UPDT", "chunks_data")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
event_count = int(events.count())

if event_count:
    desired_partitions = min(80, max(1, math.ceil(event_count / 200)))
    processing_events = events.repartition(
        desired_partitions, "EVENT_ID", "ADC_UPDT"
    )
    raw = (
        processing_events
        .select(process_blob_udf("EVENT_ID", "ADC_UPDT", "chunks_data").alias("r"))
        .select("r.*")
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    materialized_count = int(raw.count())
else:
    raw = spark.createDataFrame([], _udf_schema).persist(StorageLevel.MEMORY_AND_DISK)
    materialized_count = 0

if materialized_count != event_count:
    raise RuntimeError(
        f"Shard {SHARD_ID} selected {event_count} rows but materialized "
        f"{materialized_count}"
    )

target_columns = [
    "EVENT_ID", "VALID_UNTIL_DT_TM", "VALID_FROM_DT_TM", "UPDT_DT_TM",
    "UPDT_ID", "UPDT_TASK", "UPDT_CNT", "UPDT_APPLCTX", "LAST_UTC_TS",
    "ADC_UPDT", "BLOB_BINARY", "CONTENT_TYPE", "ENCODING", "BLOB_TEXT",
    "BINARY_SIZE", "TEXT_LENGTH", "STATUS", "anon_text", "ENCNTR_ID", "Trust",
    "raw_sha256", "decompressor_version", "parser_version",
    "post_processor_version",
]
batch_df = raw.select(*target_columns, "metrics")
history_df = raw.select(
    "EVENT_ID", "ADC_UPDT", "raw_sha256", "decompressor_version",
    "parser_version", "post_processor_version",
    F.col("history_status").alias("status"),
    "truncation_flag", "truncation_reason", "ftfy_explain",
    "decompression_strategy", F.lit(RUN_ID).alias("run_id"),
    F.lit("forward").alias("source"), "parse_status", "metrics",
)

merge_output(
    batch_df,
    batch_table,
    "t.EVENT_ID = s.EVENT_ID AND t.ADC_UPDT <=> s.ADC_UPDT",
)
merge_output(
    history_df,
    history_table,
    "t.EVENT_ID = s.EVENT_ID AND t.ADC_UPDT <=> s.ADC_UPDT "
    "AND t.run_id = s.run_id AND t.source = s.source",
)

if event_count:
    selected_keys = events.select("EVENT_ID", "ADC_UPDT").distinct()
    (
        DeltaTable.forName(spark, WORKLIST_TABLE)
        .alias("t")
        .merge(
            selected_keys.alias("s"),
            "t.EVENT_ID = s.EVENT_ID AND t.ADC_UPDT <=> s.ADC_UPDT",
        )
        .whenMatchedUpdate(set={
            "status": "'completed'",
            "process_ts": "current_timestamp()",
            "error_msg": "NULL",
        })
        .execute()
    )

raw.unpersist()
events.unpersist()

remaining_in_shard = int(
    spark.table(WORKLIST_TABLE)
    .filter(F.col("status") != "completed")
    .filter(shard_predicate)
    .count()
)

dbutils.notebook.exit(json.dumps({
    "status": "success",
    "shard_id": SHARD_ID,
    "selected_events": event_count,
    "remaining_in_shard": remaining_in_shard,
    "batch_table": batch_table,
    "history_table": history_table,
}, sort_keys=True))